# Results Writeup
Final model comparison and analysis for the portfolio README.

In [ ]:
import mlflow, pandas as pd, numpy as np, matplotlib.pyplot as plt
import sys; sys.path.insert(0,"..")
from src.config import MLFLOW_TRACKING_URI, FEATURES_PATH, TARGET_COL
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
%matplotlib inline

## 1. Overall Metric Comparison

In [ ]:
client = mlflow.tracking.MlflowClient()
exps = {"Rolling HV":"baseline_rolling","GARCH(1,1)":"baseline_garch","LightGBM":"lgbm","LSTM":"lstm"}
rows = []
for name, exp_name in exps.items():
    e = client.get_experiment_by_name(exp_name)
    if e is None: continue
    runs = client.search_runs([e.experiment_id], filter_string="tags.final = 'true'", order_by=["start_time DESC"], max_results=1)
    if not runs: continue
    m = runs[0].data.metrics
    rows.append({"Model":name, "QLIKE":m.get("test_qlike",None), "RMSE":m.get("test_rmse_log_vol",None), "MAE":m.get("test_mae_log_vol",None)})
df = pd.DataFrame(rows).set_index("Model")
df.style.highlight_min(axis=0, color="lightgreen")

## 2. QLIKE by Vol Regime

In [ ]:
regime_data = {}
for name, exp_name in exps.items():
    e = client.get_experiment_by_name(exp_name)
    if e is None: continue
    runs = client.search_runs([e.experiment_id], filter_string="tags.final = 'true'", max_results=1)
    if not runs: continue
    m = runs[0].data.metrics
    regime_data[name] = [m.get(f"test_qlike_q{q}", None) for q in range(5)]
regime_df = pd.DataFrame(regime_data, index=[f"Q{i}" for i in range(5)])
regime_df.plot.bar(figsize=(10,5), title="QLIKE by vol regime (Q0=low, Q4=high)")
plt.ylabel("QLIKE"); plt.xticks(rotation=0); plt.tight_layout()

## 3. Bootstrap CI on QLIKE Difference

In [ ]:
for ml in ["LightGBM","LSTM"]:
    e = client.get_experiment_by_name(exps[ml])
    if e is None: continue
    runs = client.search_runs([e.experiment_id], filter_string="tags.final = 'true'", max_results=1)
    if not runs: continue
    m = runs[0].data.metrics
    pt = m.get("test_qlike_diff_vs_baseline_point","N/A")
    lo = m.get("test_qlike_diff_vs_baseline_ci_low","N/A")
    hi = m.get("test_qlike_diff_vs_baseline_ci_high","N/A")
    print(f"{ml}: delta={pt:.4f} 95%CI=[{lo:.4f}, {hi:.4f}]")

## 4. Limitations & Next Steps

- **Survivorship bias**: current S&P 500 only; excludes ~30% of tickers that existed in 2014 but were later removed.
- **COVID in val**: 2020-2021 vol spike is in validation, not test. Models may overfit to crisis regimes.
- **Phase 1 next**: LLM-extracted news/earnings sentiment features via Claude API.
